# Notebook 017: Recursive Schema Traversal: Flattening Deeply Nested JSON/Structs

**Module:** Module 02: Complex Data Types & Nested Structures Handling  
**Difficulty:** `Advanced`  
**Databricks Masterclass Reference:** Video 11: Auto-loader & Video 23: Interview Questions  

---

## 1. Business Scenario & Real-World Context
An enterprise integration hub ingests complex payloads from ERP and CRM systems containing 5+ levels of nested structs and arrays. Analysts require a completely flattened tabular DataFrame where column names reflect their full dot-delimited path.

## 2. Technical Requirement & Problem Statement
Develop a generic, production-ready recursive schema flattener in Python that traverses any arbitrarily deep Spark schema. The function must explode nested arrays and flatten all nested structs into top-level columns (`parent_child_field`).

## 3. Spark & Databricks Architecture Notes
Deeply nested schemas can cause severe query compilation overhead. Traversing the schema programmatically inspects `df.schema.fields` and constructs a projection expression list dynamically without executing any JVM actions.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_017_Recursive_Schema_Traversal_Flattening_Deeply_Nested_JSON/Structs")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Generating Arbitrarily Deep Nested Schema
Create a multi-level nested customer record with structs inside structs.

In [ ]:
deep_data = [
    {
        "id": "ACC-99",
        "company": {
            "name": "Acme Corp",
            "headquarters": {
                "geo": {"lat": 37.7749, "lon": -122.4194},
                "city": "San Francisco"
            }
        },
        "metrics": {"financials": {"revenue": 5000000.0, "ebitda": 1200000.0}}
    }
]

import json
rdd_json = spark.sparkContext.parallelize([json.dumps(d) for d in deep_data])
df_deep = spark.read.json(rdd_json)
print("Original Deeply Nested Schema:")
df_deep.printSchema()

## 6. Recursive Flattening Algorithm
Implement recursive schema inspection function.

In [ ]:
def flatten_dataframe(df):
    """
    Recursively flattens all StructType columns in a DataFrame into top-level columns.
    """
    flat_cols = []
    has_struct = False
    
    for field in df.schema.fields:
        if isinstance(field.dataType, T.StructType):
            has_struct = True
            for child in field.dataType.fields:
                flat_cols.append(
                    F.col(f"{field.name}.{child.name}").alias(f"{field.name}_{child.name}")
                )
        else:
            flat_cols.append(F.col(field.name))
            
    df_result = df.select(*flat_cols)
    # If any struct columns were flattened, recurse until all structs are resolved
    if has_struct:
        return flatten_dataframe(df_result)
    return df_result

df_flattened = flatten_dataframe(df_deep)
print("Completely Flattened Schema:")
df_flattened.printSchema()
df_flattened.show(truncate=False)

## 7. Validation & Column Verification
Verify all leaf columns are top-level primitives.

In [ ]:
expected_cols = [
    "id",
    "company_name",
    "company_headquarters_city",
    "company_headquarters_geo_lat",
    "company_headquarters_geo_lon",
    "metrics_financials_ebitda",
    "metrics_financials_revenue"
]

for c in expected_cols:
    assert c in df_flattened.columns, f"Expected column {c} missing from flattened DataFrame"

print(f"Successfully flattened deep schema into {len(df_flattened.columns)} scalar columns!")

## 9. Key Architectural Takeaways
- **Programmatic schema inspection via `df.schema.fields` enables dynamic metadata-driven pipelines.**
- **Recursive flattening eliminates manual column renaming and hardcoded nested selectors.**
- **Flattened schemas allow downstream BI tools (Power BI, Tableau) to query data without JSON parsing.**

## 10. Senior Data Engineering Interview Questions
1. How do you handle nested `ArrayType(StructType)` columns during automated schema flattening?
2. What are the performance implications of flattening a schema with 500+ leaf columns in Parquet?
3. Why can recursive schema transformations exceed Java bytecode limits (64KB method size) in Spark?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
